### Allscripts Sunrise (SCM) Procedure Occurrence Hydration

This notebook currently derives procedures from Sunrise orders/tasks so `procedure_occurrence` is no longer a commented-out template.

Client follow-up also provided a separate SCM billing extract pattern in `(Clone) procedures_SCM.py` that sources CPT/HCPCS-style procedures from Soarian, DSS, and Athena `omny_accounts` feeds through an SCM encounter mapper. Treat that billing flow as complementary source context that still needs reconciliation with this OMOP hydration path and its concept-mapping strategy.

In [0]:
%sql
-- Reset Gold
TRUNCATE TABLE _exponent.omop_scm.procedure_occurrence;


In [0]:
%sql
-- Reset Silver
DELETE FROM _exponent.omop_silver.procedure_occurrence
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- Reset Mapping
DELETE FROM _exponent.omop_mapping.source_to_procedure_occurrence
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW silver_procedure_occurrence AS
WITH concept_relationship_maps_to_procedure_dedup AS (
  SELECT
    cr.concept_id_1 AS source_concept_id,
    cr.concept_id_2 AS standard_concept_id,
    ROW_NUMBER() OVER (
      PARTITION BY cr.concept_id_1
      ORDER BY cr.concept_id_2 ASC
    ) AS rn
  FROM _exponent.omop.concept_relationship cr
  INNER JOIN _exponent.omop.concept target_concept
    ON target_concept.concept_id = cr.concept_id_2
   AND target_concept.standard_concept = 'S'
   AND target_concept.invalid_reason IS NULL
   AND target_concept.domain_id = 'Procedure'
  WHERE cr.relationship_id = 'Maps to'
    AND cr.invalid_reason IS NULL
), procedure_mapping AS (
  SELECT
    source_concept.concept_id AS source_concept_id,
    source_concept.concept_code AS source_concept_code,
    COALESCE(mapped.standard_concept_id, source_concept.concept_id) AS standard_concept_id
  FROM _exponent.omop.concept source_concept
  LEFT JOIN concept_relationship_maps_to_procedure_dedup mapped
    ON mapped.source_concept_id = source_concept.concept_id
   AND mapped.rn = 1
  WHERE source_concept.vocabulary_id IN ('CPT4', 'HCPCS')
    AND source_concept.domain_id = 'Procedure'
    AND source_concept.invalid_reason IS NULL
), staged AS (
  SELECT
    CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3order', 'GUID', CAST(ord.GUID AS STRING)) AS procedure_occurrence_source_value,
    stp.person_id,
    COALESCE(procedure_mapping.standard_concept_id, proc_concept.omop_concept_id, 0) AS procedure_concept_id,
    CAST(COALESCE(oto.PerformedFromDtm, ord.PerformedDtm, ord.SignificantDtm, ord.RequestedDtm, ord.Entered, ord.CreatedWhen) AS DATE) AS procedure_date,
    COALESCE(oto.PerformedFromDtm, ord.PerformedDtm, ord.SignificantDtm, ord.RequestedDtm, ord.Entered, ord.CreatedWhen) AS procedure_datetime,
    32817 AS procedure_type_concept_id,
    COALESCE(mod_concept.omop_concept_id, 0) AS modifier_concept_id,
    CAST(1 AS DOUBLE) AS quantity,
    stpr.provider_id AS provider_id,
    stvo.visit_occurrence_id AS visit_occurrence_id,
    NULL AS visit_detail_id,
    COALESCE(NULLIF(TRIM(ord.IDCode), ''), NULLIF(TRIM(ord.Name), ''), NULLIF(TRIM(oto.TaskName), ''), ord.TypeCode) AS procedure_source_value,
    COALESCE(procedure_mapping.source_concept_id, 0) AS procedure_source_concept_id,
    NULLIF(TRIM(ord.Modifier), '') AS modifier_source_value,
    'allscripts_scm' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp
  FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
  LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3ordertaskoccurrence oto
    ON oto.OrderGUID = ord.GUID
   AND oto.Active = TRUE
  INNER JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(COALESCE(oto.ClientGUID, ord.ClientGUID) AS STRING))
   AND stp.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_provider stpr
    ON stpr.provider_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3careprovider', 'GUID', CAST(COALESCE(oto.PerformedProviderGUID, oto.EnteredProviderGUID, ord.CareProviderGUID) AS STRING))
   AND stpr.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3clientvisit', 'GUID', CAST(ord.ClientVisitGUID AS STRING))
   AND stvo.source_system = 'allscripts_scm'
   AND stvo.active_flag = TRUE
  LEFT JOIN procedure_mapping
    ON procedure_mapping.source_concept_code = NULLIF(REGEXP_REPLACE(CAST(ord.IDCode AS STRING), '[\s\u00A0]+', ''), '')
  LEFT JOIN _exponent.omop_mapping.domain_source_to_concept proc_concept
    ON proc_concept.domain_id = 'Procedure'
   AND proc_concept.source_system = 'allscripts_scm'
   AND proc_concept.source_id = COALESCE(NULLIF(TRIM(ord.IDCode), ''), NULLIF(TRIM(ord.Name), ''), NULLIF(TRIM(oto.TaskName), ''), ord.TypeCode)
  LEFT JOIN _exponent.omop_mapping.domain_source_to_concept mod_concept
    ON mod_concept.domain_id = 'Modifier'
   AND mod_concept.source_system = 'allscripts_scm'
   AND mod_concept.source_id = NULLIF(TRIM(ord.Modifier), '')
  WHERE ord.Active = TRUE
    AND ord.GUID IS NOT NULL
    AND COALESCE(oto.ClientGUID, ord.ClientGUID) IS NOT NULL
    AND COALESCE(oto.PerformedFromDtm, ord.PerformedDtm, ord.SignificantDtm, ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL
    AND UPPER(COALESCE(ord.OrderStatusCode, oto.TaskStatusCode, '')) NOT IN ('CAN', 'CANCELLED', 'CANCELED')
    AND UPPER(COALESCE(ord.TypeCode, '')) NOT IN ('MEDICATION', 'MED', 'PHARMACY', 'LAB', 'LABORATORY')
), deduped AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY procedure_occurrence_source_value ORDER BY procedure_datetime DESC) AS rn
  FROM staged
)
SELECT
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
FROM deduped
WHERE rn = 1;

In [0]:
%sql
INSERT INTO _exponent.omop_silver.procedure_occurrence (
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
)
SELECT
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
FROM silver_procedure_occurrence;


In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_procedure_occurrence (
  source_system,
  procedure_occurrence_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp
)
SELECT
  s.source_system,
  s.procedure_occurrence_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP())
FROM silver_procedure_occurrence s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_procedure_occurrence x
  ON s.procedure_occurrence_source_value = x.procedure_occurrence_source_value
 AND x.source_system = 'allscripts_scm';


In [0]:
%sql
INSERT INTO _exponent.omop_scm.procedure_occurrence (
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value
)
SELECT
  spo.procedure_occurrence_id,
  s.person_id,
  s.procedure_concept_id,
  s.procedure_date,
  s.procedure_datetime,
  s.procedure_type_concept_id,
  s.modifier_concept_id,
  s.quantity,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.procedure_source_value,
  s.procedure_source_concept_id,
  s.modifier_source_value
FROM silver_procedure_occurrence s
JOIN _exponent.omop_mapping.source_to_procedure_occurrence spo
  ON spo.procedure_occurrence_source_value = s.procedure_occurrence_source_value
 AND spo.source_system = 'allscripts_scm'
 AND spo.active_flag = TRUE;
